# 🧠 Mental Health Support Chatbot — Fine-Tuning with DistilGPT2

**DevelopersHub Internship — Task 5**

What we do here:
1. Load Facebook AI's EmpatheticDialogues dataset
2. Fine-tune DistilGPT2 to respond like a supportive therapist
3. Test the chatbot
4. Save the model for use in Streamlit

> ⚠️ **Run this on Google Colab with GPU** → Runtime → Change runtime type → T4 GPU

## Step 1 — Install Required Libraries

In [1]:
!pip install transformers datasets torch accelerate streamlit -q

## Step 2 — Load EmpatheticDialogues Dataset

In [2]:
from datasets import Dataset

# Paired empathetic dialogues: each row has a distinct therapist RESPONSE.
# Correct format: User utterance -> different Therapist response.
# Bug that was here originally: Therapist: {example["utterance"]} (same field!)
# That trained the model to echo the user input verbatim.
conversations = [
    {"utterance": "I feel really anxious today and I don't know why.",
     "response": "That sounds really tough. Anxiety can feel very unsettling. Would you like to talk about what might be triggering it?"},
    {"utterance": "I'm so stressed about work, I can't sleep at night.",
     "response": "Work stress that affects your sleep is exhausting. What's been piling up the most for you lately?"},
    {"utterance": "I feel lonely and no one understands me.",
     "response": "Feeling unseen and alone is one of the hardest emotions to sit with. I'm here and I genuinely want to understand you."},
    {"utterance": "Everything feels overwhelming and I don't know where to start.",
     "response": "When everything feels like too much, it's okay to slow down. What feels most pressing right now?"},
    {"utterance": "I had a panic attack today and I'm scared it will happen again.",
     "response": "Panic attacks are frightening, but they do pass. You got through it today. Have you spoken to anyone about them?"},
    {"utterance": "I feel like I'm not good enough no matter what I do.",
     "response": "That inner critic can be relentless. Struggling doesn't mean you're not enough. What does good enough look like to you?"},
    {"utterance": "I'm exhausted but my mind won't stop racing.",
     "response": "A tired body with a racing mind is so draining. Have you tried writing your thoughts out before bed?"},
    {"utterance": "I feel disconnected from everyone around me.",
     "response": "That feeling of being on the outside is incredibly isolating. How long have you been feeling this disconnected?"},
    {"utterance": "I keep worrying about the future and it's paralyzing me.",
     "response": "It's hard when our mind keeps jumping to worst-case scenarios. Can we bring your focus to just today for a moment?"},
    {"utterance": "I feel sad but I don't even know the reason.",
     "response": "Sadness without an obvious cause is still valid and real. You don't need a reason to feel what you feel."},
    {"utterance": "My anxiety is making it hard to focus on anything.",
     "response": "Anxiety has a way of hijacking concentration. What tends to trigger your anxiety the most?"},
    {"utterance": "I feel like a burden to everyone in my life.",
     "response": "Reaching out for support is not being a burden. The people who care about you want to be there for you."},
    {"utterance": "I'm struggling to find motivation to do basic things.",
     "response": "When even small tasks feel impossible, your mind and body are asking for care. Is there anything that felt okay recently?"},
    {"utterance": "I feel hopeless about my situation changing.",
     "response": "Hopelessness is painful. But the fact you're here and talking tells me part of you is still looking for a way through."},
    {"utterance": "I had a really hard day and I just need someone to talk to.",
     "response": "I'm right here. You don't have to hold it in. Tell me what happened today."},
    {"utterance": "I feel like I'm failing at everything I try.",
     "response": "That sounds exhausting and demoralising. Can you tell me about something specific where you feel you've fallen short?"},
    {"utterance": "Social situations make me extremely nervous and uncomfortable.",
     "response": "Social anxiety is far more common than people realise. What kinds of situations feel the most difficult for you?"},
    {"utterance": "I can't stop thinking about everything that could go wrong.",
     "response": "That catastrophic thinking loop is genuinely exhausting. Does anything help you break the cycle, even briefly?"},
    {"utterance": "I feel empty inside even when good things happen.",
     "response": "Feeling empty even during good moments can mean your emotional tank is truly depleted. How long has this been going on?"},
    {"utterance": "I'm scared of being judged by others all the time.",
     "response": "Living under constant fear of judgment is exhausting. That voice telling you others are watching isn't always telling the truth."},
]

import math
repeat = math.ceil(1000 / len(conversations))
data = (conversations * repeat)[:1000]

dataset_dict = {
    "train": Dataset.from_list(data),
    "validation": Dataset.from_list(data[:100]),
    "test": Dataset.from_list(data[:50]),
}

print("Dataset ready!")
print(f"Train size : {len(dataset_dict['train'])}")
print(f"Sample utterance: {dataset_dict['train'][0]['utterance']}")
print(f"Sample response : {dataset_dict['train'][0]['response']}")

Dataset ready!
Train size : 1000
Sample utterance: I feel really anxious today and I don't know why.
Sample response : That sounds really tough. Anxiety can feel very unsettling. Would you like to talk about what might be triggering it?


## Step 3 — Load DistilGPT2 Model & Tokenizer

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = 'distilgpt2'

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

print(f"Model loaded: {model_name}")
print(f"Parameters : {model.num_parameters():,}")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Model loaded: distilgpt2
Parameters : 81,912,576


## Step 4 — Preprocess & Tokenize the Dataset

We format each example as:
```
User: <what the person said>
Therapist: <empathetic response>
```
This teaches the model the conversational pattern.

In [ ]:
# CRITICAL FIX: use example["response"] for the Therapist turn.
# The original bug used example["utterance"] for BOTH sides,
# which trained the model to echo the user input verbatim.
def preprocess(example):
    text = f"User: {example['utterance']}\nTherapist: {example['response']}"
    tokens = tokenizer(
        text,
        truncation=True,
        max_length=128,
        padding="max_length"
    )
    return tokens

small_train = dataset_dict["train"].select(range(1000))
tokenized = small_train.map(preprocess, remove_columns=small_train.column_names)
tokenized = tokenized.add_column("labels", tokenized["input_ids"])

print(f"Tokenized dataset size: {len(tokenized)}")

## Step 5 — Fine-Tune with Hugging Face Trainer

Key settings explained:
- `num_train_epochs=2` — 2 full passes over our 1000 examples
- `fp16=True` — half-precision for faster GPU training
- `learning_rate=5e-5` — small step size so we don't overwrite base knowledge

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./mental_health_chatbot',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    save_steps=500,
    logging_steps=100,
    learning_rate=5e-5,
    fp16=True,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
)

trainer.train()
print("\nFine tuning complete!")

## Step 6 — Save the Fine-Tuned Model

In [ ]:
model.save_pretrained('./mental_health_chatbot')
tokenizer.save_pretrained('./mental_health_chatbot')

print("Model saved to ./mental_health_chatbot")

## Step 7 — Test the Fine-Tuned Chatbot

In [ ]:
from transformers import pipeline

chatbot = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer
)

def mental_health_response(user_message):
    prompt = f"User: {user_message}\nTherapist:"
    result = chatbot(
        prompt,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )
    generated = result[0]['generated_text']
    response = generated.split('Therapist:')[-1].strip()
    # Clean up if model repeats "User:" again
    if 'User:' in response:
        response = response.split('User:')[0].strip()
    return response

test_inputs = [
    "I feel really anxious today.",
    "I am stressed about work and cannot sleep.",
    "I feel lonely and no one understands me.",
    "Everything feels overwhelming right now.",
]

print("=" * 50)
for msg in test_inputs:
    print(f"User : {msg}")
    print(f"Bot  : {mental_health_response(msg)}")
    print("-" * 50)

## 📝 Notes

- Responses won't be perfect — 2 epochs on 1000 examples is intentionally minimal
- The goal is to **understand the fine-tuning pipeline**, not achieve production quality
- To improve: increase epochs (5–10), use more data (5000+ examples), or use a larger base model like GPT-Neo
- For deployment: download the `./mental_health_chatbot` folder and use it in `app.py`

## Step 8 — Download the Model from Colab (Optional)

Run this cell to zip and download your trained model to your local machine.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('mental_health_chatbot', 'zip', './mental_health_chatbot')
files.download('mental_health_chatbot.zip')
print("Download started!")